#Understanding Retrieval-Augmented Generation (RAG)
## What is RAG?
Retrieval-Augmented Generation (RAG) is an advanced technique in the field of Natural Language Processing (NLP) that combines two powerful components:

1. Retrieval Systems: These are mechanisms for fetching relevant data from a large corpus or database. For example, searching for specific pieces of information in documents or datasets.
2. Generative Models: These are AI models, such as OpenAI's GPT, that can generate coherent and meaningful responses or content based on given input.


RAG bridges the gap between structured information retrieval and the generative capabilities of modern AI models, making it possible to generate contextually rich and accurate responses.

This notebook serves as an introduction to RAG and demonstrates how it can be used for simple Question-Answering (QA) tasks. It walks you through the process of:

* Uploading and processing documents.
* Splitting documents into manageable chunks for retrieval.
* Using a generative AI model to answer questions based on the retrieved content.

By the end of this notebook, you will understand the basic workings of a RAG system and see how it can be applied in real-world scenarios, such as document-based Q&A. Let's get started!

# Install necessary libraries
This cell installs the required libraries for the program to function.

In [ ]:
!pip install openai langchain langchain-core langchain-community langchain-openai langchain-text-splitters faiss-cpu numpy tiktoken PyPDF2 python-docx
from IPython.display import clear_output
clear_output()

# Import required modules
This cell imports all the necessary modules and packages used throughout the program.

In [ ]:
import os
from typing import List
import numpy as np
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from google.colab import files
from PyPDF2 import PdfReader
from docx import Document
from langchain_core.documents import Document as LangchainDocument

# Initializing OpenAI Components
This section of code sets up the essential components needed to build a Retrieval-Augmented Generation (RAG) system:

* Embeddings: These are numerical representations of text that allow the system to compare and find similarities between different pieces of content. In this code, we use OpenAIEmbeddings, which leverages OpenAI's embedding models.

* Text Splitter: Long documents need to be split into smaller, manageable pieces (or chunks) for efficient processing and retrieval. The RecursiveCharacterTextSplitter is configured to:
  * Split text into chunks of up to 1,000 characters.
  * Allow an overlap of 200 characters between chunks to preserve context across splits.

In [ ]:
def initialize_openai(api_key):
    embeddings = OpenAIEmbeddings(openai_api_key=api_key)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    return embeddings, text_splitter

api_key = "" # Please enter your OpenAI API key
embeddings, text_splitter = initialize_openai(api_key)

# Uploading and Loading Documents
This section handles the process of uploading and processing documents to prepare them for retrieval and question-answering. Here's how it works:

* File Upload: The system prompts the user to upload files in supported formats (PDF, DOCX, or TXT).

* Processing Files:

  * PDF Files: Uses the PdfReader to extract text from each page.
  * DOCX Files: Uses the Document class to read text from each paragraph.
  * TXT Files: Reads the entire content of the file as plain text.
  * If the file format is unsupported, it is skipped with a warning message.
  * Document Creation: Each file's content is wrapped into a LangchainDocument object, which includes metadata like the source filename.

* Splitting Documents: The documents are divided into smaller chunks using the text splitter. This ensures that retrieval and question-answering are efficient and accurate.

* Vector Store Management:

  * If no vector store exists, a new one is created using the processed chunks and embeddings.
  * If an existing vector store is provided, the new chunks are added to it.
* Output: The function updates the vector store and informs the user about the number of processed chunks and documents.

In [ ]:
vector_store = None # We create an empty vector store, so we can append to it later

In [ ]:
def upload_and_load_documents(text_splitter, embeddings, vector_store=None):
    print("Please upload your files (PDF, DOCX, or TXT)...")
    uploaded = files.upload()
    documents = []

    for filename, content in uploaded.items():
        temp_path = os.path.join(filename)
        with open(temp_path, 'wb') as f:
            f.write(content)

        ext = os.path.splitext(filename)[1].lower()

        try:
            if ext == '.pdf':
                pdf_reader = PdfReader(temp_path)
                text = ''.join(page.extract_text() for page in pdf_reader.pages)
            elif ext == '.docx':
                doc = Document(temp_path)
                text = '\n'.join(paragraph.text for paragraph in doc.paragraphs)
            elif ext == '.txt':
                with open(temp_path, 'r', encoding='utf-8') as file:
                    text = file.read()
            else:
                print(f"Unsupported file type for {filename}. Skipping.")
                continue

            documents.append(LangchainDocument(page_content=text, metadata={"source": filename}))
        except Exception as e:
            print(f"Failed to process {filename}: {e}")

    if not documents:
        print("No documents processed. Exiting.")
        return vector_store

    chunks = text_splitter.split_documents(documents)

    if vector_store is None:
        # Create a new vector store if none exists
        vector_store = FAISS.from_documents(chunks, embeddings)
        print(f"Created a new vector store with {len(chunks)} chunks from {len(uploaded)} documents.")
    else:
        # Add new documents to the existing vector store
        vector_store.add_documents(chunks)
        print(f"Appended {len(chunks)} chunks from {len(uploaded)} documents to the existing vector store.")

    return vector_store

vector_store = upload_and_load_documents(text_splitter, embeddings, vector_store)

Please upload your files (PDF, DOCX, or TXT)...


Saving KA Specializations.docx to KA Specializations.docx
Appended 20 chunks from 1 documents to the existing vector store.


# Querying the System
This function allows the user to ask a question and retrieves relevant answers based on the uploaded and processed documents. Here's a breakdown of how it works:

* Check Vector Store:

  * Ensures that a vector store (containing document embeddings) is loaded. If not, the function prompts the user to load documents first.
* Find Relevant Documents:

  * The system performs a similarity search in the vector store to find the top 3 document chunks that are most relevant to the user's question.
* Create a Prompt:

  * Combines the retrieved document content into a "context."
  * Forms a question-answering prompt for the AI model by including the context and the user's question.
* Interact with OpenAI:

  * Uses the OpenAI API to send the prompt to the GPT model (gpt-4o in this case).
  * The model generates an answer based on the provided context.
* Output the Answer:

  * Returns the AI-generated response and displays it to the user.

In [ ]:
def query_system(question, vector_store, api_key):
    if not vector_store:
        return "Please load documents first."

    relevant_docs = vector_store.similarity_search(question, k=3)
    context = "\n".join(doc.page_content for doc in relevant_docs)
    print(f"Context is {context}")
    print("Answer:")


    prompt = f"""Given the following context, answer the question:

    Context:
    {context}

    Question: {question}"""

    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=500
    )
    return response.choices[0].message.content

question = input("Please enter your question: ") # Enter your question
answer = query_system(question, vector_store, api_key)
print(f"Answer: {answer}")

Context is KAUST Academy Specialization in Artificial intelligence
Introduction:
KAUST Academy is proposing to design a specialization program in Artificial AI). This program aims to empower students and fresh graduates studying or that are interested in working in the AI sector by providing them with advanced technical knowledge, leadership skills, and valuable industry experience.
Program Structure:
Phase 1: Online Learning (8 weeks) - Coursera Platform
Target audience: All graduate students and undergraduate students near completion of their degrees in the Kingdom working in the 
Delivery: Online modules hosted on Coursera, a user-friendly learning platform.
Content (12 Courses, including): 
Python 3 Programming Specialization
Mathematics for Machine Learning and Data Science Specialization
Improving Deep Neural Networks: Hyperparameter Tuning, Regularization and Optimization
Exploratory Data Analysis for Machine Learning
Supervised Machine Learning: Regression
KAUST Academy Special